# LegalQA Task 2 — Kaggle Dual-T4 Small-Train Probe
Bounded real-SFT validation (200 examples, 60 steps, held-out fold 0) proving the CUDA training path: data join, optimizer steps, VRAM stability, adapter save/strict-reload, single-query generation. Makes no quality or promotion claims.

In [ ]:
# Cell 1: Environment & Guardrails
import os, sys

# Guardrails for CUDA allocator, Transformers, and non-Torch frameworks
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".05"
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TORCH"] = "1"
print("Transformers async model loading: DISABLED for T4-safe QLoRA load")
print("PyTorch allocator configured with expandable_segments:True,max_split_size_mb:128")
print("JAX/TensorFlow GPU memory preallocation: DISABLED")

SEED = 42
CONFIG_PATH = "configs/task2/runtime/kaggle_t4x2.yaml"
LEGACY_CONFIG_PATH = "configs/kaggle_smoke_t4.yaml"
print(f"Config target: {CONFIG_PATH}")


In [ ]:
# Cell 2: Hardware & Device Allocation
import os, sys, subprocess, torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Kaggle GPU execution but torch.cuda.is_available() is False.")

gpu_count = torch.cuda.device_count()
print(f"CUDA GPUs Detected: {gpu_count}")
for i in range(gpu_count):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.1f} GB | Compute: sm_{p.major}{p.minor}")

try:
    print("=== Initial NVIDIA-SMI Telemetry ===")
    print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception as e:
    print(f"nvidia-smi check skipped: {e}")

GEN_DEVICE = "cuda:0"
RETRIEVAL_DEVICE = "cuda:1" if gpu_count >= 2 else "cuda:0"
print(f"Hardware Allocation -> Generator: {GEN_DEVICE} | Retrieval/Reranker: {RETRIEVAL_DEVICE}")


In [ ]:
# Cell 3: Code Root & Workspace Bootstrap
from pathlib import Path
import os, sys, subprocess, json

code_root = os.path.abspath("LegalQA")
if not os.path.isdir(code_root):
    print("Cloning LegalQA repository from GitHub...")
    subprocess.check_call(["git", "clone", "https://github.com/silent9669/LegalQA.git"])

# Fetch all branches and check out main or candidate
subprocess.check_call(["git", "-C", code_root, "fetch", "origin"])

# Look for mounted candidate manifest
candidate_file = Path("/kaggle/input/legalqa-candidate/candidate_manifest.json")
if not candidate_file.exists():
    cands = list((Path(code_root) / "artifacts" / "candidates").glob("*/candidate_manifest.json"))
    if cands:
        candidate_file = sorted(cands, key=lambda p: p.stat().st_mtime)[-1]


# Robust mount resolution (flat + nested layouts, lazy sync)
if code_root not in sys.path:
    sys.path.insert(0, code_root)
if not candidate_file.exists():
    try:
        from src.task2.kaggle_mount import find_candidate_manifest
        candidate_file = find_candidate_manifest(timeout_seconds=120)
    except Exception as _e:
        print(f'mount search unavailable: {_e}')


# Robust candidate resolution: wait for lazy mount sync, then search.
# Never fall back to main: an unresolved candidate is a hard failure.
import time as _time
_deadline = _time.monotonic() + 180
while not candidate_file.exists():
    _hits = [p for p in Path('/kaggle/input').rglob('candidate_manifest.json') if p.is_file()] if os.path.isdir('/kaggle/input') else []
    if _hits:
        candidate_file = sorted(_hits)[0]
        print(f'candidate manifest located by mount search: {candidate_file}')
        break
    if _time.monotonic() >= _deadline:
        raise RuntimeError('candidate_manifest.json not mounted after 180s; refusing main-branch fallback')
    _time.sleep(5)

if candidate_file.exists():
    cand_data = json.loads(candidate_file.read_text(encoding="utf-8"))
    cand_sha = cand_data.get("git_commit_sha")
    print(f"Checking out exact candidate SHA: {cand_sha}...")
    subprocess.check_call(["git", "-C", code_root, "checkout", "--detach", cand_sha])
else:
    print("Notice: Checking out main...")
    subprocess.check_call(["git", "-C", code_root, "checkout", "main"])

head_rev = subprocess.check_output(["git", "-C", code_root, "rev-parse", "HEAD"], text=True).strip()
print(f"Active Git Commit: {head_rev}")

if code_root not in sys.path:
    sys.path.insert(0, code_root)
print(f"Active Code Root: {code_root}")
# Probe diagnosis: list mounted inputs
import os as _os
print('kaggle input tree:', sorted(str(q) for q in Path('/kaggle/input').rglob('*') if q.is_dir())[:20] if _os.path.isdir('/kaggle/input') else 'NO /kaggle/input')



In [ ]:
# Cell 4: Dependency Bootstrap & Runtime Verification
import subprocess
import scripts.bootstrap_kaggle_env as bstrap
from src.common.env_loader import load_environment

bstrap.print_preinstalled_environment()
BOOTSTRAP_RESULT = bstrap.bootstrap_dependencies()
bstrap.verify_runtime_imports(strict=True)

# Load environment credentials (.env, Kaggle secrets, or env vars)
env_status = load_environment()
print(f"Environment Loaded: {env_status.get('loaded_from_file') or 'default/secrets'}")
print(f"Hugging Face Auth: {'CONFIGURED (' + env_status['hf_token_masked'] + ')' if env_status['hf_token_configured'] else 'NOT CONFIGURED'}")
print(f"Kaggle Auth: {'CONFIGURED (' + env_status['kaggle_user'] + ')' if env_status['kaggle_configured'] else 'NOT CONFIGURED'}")
print("Dependency bootstrap complete: all user-space packages verified.")

try:
    print("=== Post-Bootstrap NVIDIA-SMI ===")
    print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception:
    pass

In [ ]:
# Cell 5: Dataset Mount & Integrity Verification
import subprocess
from src.task2.path_resolver import resolve_runtime_paths
from src.task2.dataset.validator import validate_dataset

paths = resolve_runtime_paths("/kaggle/input", strict=False)
print(f"Resolved Dataset Root: {paths.get('runtime_root')}")

schema_file = os.path.join(code_root, "configs/dataset_schema.yaml")
val_report = validate_dataset(data_dir=paths["runtime_root"], schema_path=schema_file)
print(f"Dataset Manifest Verified: {val_report.get('manifest_verified')} (Status: {val_report.get('status')})")
if val_report.get("status") != "PASS":
    raise RuntimeError(f"Dataset validation failed: {val_report.get('errors')}")

try:
    print("=== Post-Dataset NVIDIA-SMI ===")
    print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception:
    pass


In [ ]:
# Cell 6: Bounded Small-Scale QLoRA Training Probe
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
from scripts.kaggle_train_probe import run_train_probe
probe_report = run_train_probe(
    candidate_path=str(candidate_file),
    data_dir=paths["runtime_root"],
    output_dir="/kaggle/working",
    max_steps=60,
    max_examples=200,
)
print("Probe status:", probe_report["status"])


In [ ]:
# Cell 7: Verify Train Probe Report
import json
rep = json.load(open("/kaggle/working/train_probe_report.json"))
assert rep["status"] == "PASS", rep
assert rep["optimizer_steps"] >= 60, rep
assert rep["reload"] == "pass", rep
print(f"Train probe PASS: {rep['optimizer_steps']} steps, peak {rep['peak_allocated_mb']} MB")
